In [1]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import json
from time import time
import gc
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass

random.seed(42)

from src.agents.hosted import AgentConnector, AgentConnectorConfig, AgentConnectionType, LocalAgentConnectionParams, AgentModelConfig
from src.utils import ReaderMetrics

VECTOR_DB_PATH = '../../../../data/natural_questions/dbs/v3/densedb'
CHUNKS_PATH = "../../../../data/natural_questions/natural_qa_chunked/chunked_nqa1.csv"
BASE_DATASET_PATH = "../../../../data/natural_questions/natural_q1.csv"
AGENT_MODEL_PATH = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"

/home/jovyan/work/alexander_workspace/conda/envs/rag_alex/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
PARAMS = {
    'version': 2,
    'num_samples': 2000,
    'num_contexts': 5,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- [{score}] {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. Each text has a corresponding real-value score of its relevance to the question in square brackets at the beginning. Scores are ranged from 0.0 (the text is not suitable for generating an answer based on it) to 1.0 (the text is suitable for generating an answer based on it). Use this information. Choose texts with high enough relevance scores. If, based on the specified scores, there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'scores': {'rel': 1.0, 'unrel': 0.0},
    'gen_strat': {'max_new_tokens': 1024},
    'stub_answer': "I do not have an answer to your question"
}

DB_NAME = 'natural_questions'

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'

#### Подключение к векторной бд

In [3]:
@dataclass
class EmbedderModelConfig:
    model_name_or_path: str = '../../../../models/multilingual-e5-small'
    prompts: Dict = field(default_factory=lambda: {"query": "query: ", "passage": "passage: "})
    device: str = 'cuda'
    normalize_embeddings: bool = True

class EmbedderModel:
    def __init__(self, config: EmbedderModelConfig = EmbedderModelConfig()) -> None:
        self.config = EmbedderModelConfig() if config is None else config
        self.model = SentenceTransformer(
            config.model_name_or_path, device=config.device,
            prompts=config.prompts
        )

    def encode_queries(self, queries: List[str], **kwargs) -> List[List[float]]:
        output = self.model.encode(queries, prompt_name='query', 
                                 normalize_embeddings=self.config.normalize_embeddings, **kwargs)
        return output

    def encode_passages(self, passages: List[str], **kwargs) -> List[List[float]]:
        output = self.model.encode(passages, prompt_name='query',
                                 normalize_embeddings=self.config.normalize_embeddings,
                                 **kwargs)
        return [list(obj.astype(float)) for obj in output]

In [4]:
client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
collection = client.get_collection(name=DB_NAME)
embedder = EmbedderModel()
print(collection.count())

164864


#### Подключение к агенту

In [5]:
agent_config = AgentConnectorConfig(
    agent_config = AgentModelConfig(model_name_or_path = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"))
agent = AgentConnector.open(agent_config)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [6]:
agent.generate("Сколько будет 2 + 2?")

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


'2 + 2 = 4'

### Формируем список контекстов для каждого запроса со скорами

In [7]:
exp55_gen_info = "../../only_relevant_context_without_score (exp #5.5)/natural_questions/logs/v2/generation_info.json"
with open(exp55_gen_info, 'r', encoding='utf-8') as fd:
    exp55_gen_info_json = json.loads(fd.read())

In [8]:
dataset_df = pd.read_csv(BASE_DATASET_PATH)

In [9]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    # retrieving relevant chunk
    #emb_query = embedder.encode_queries([dataset_df['question'][i]])[0]
    #output = collection.query(
    #    query_embeddings=[emb_query.tolist()],
    #    include=['metadatas'], n_results=1)

    cur_rel_doc = dataset_df['document'][i]
    #cur_list_ids = [(PARAMS['scores']['unrel'], output['metadatas'][0][0]['chunk_index'])]
    cur_list_ids = [(PARAMS['scores']['unrel'], exp55_gen_info_json[i]['used_contexts'][0][1])]
    
    while len(cur_list_ids) != PARAMS['num_contexts']:
        unrel_context_id = random.randint(0, dataset_df.shape[0]-1)

        prep_cntx = (PARAMS['scores']['unrel'], unrel_context_id)
        if dataset_df['document'][unrel_context_id] != cur_rel_doc:
            cur_list_ids.append(prep_cntx)

    CONTEXTS_LIST_IDS.append(cur_list_ids)

100%|██████████| 2000/2000 [00:00<00:00, 27757.36it/s]


In [10]:
CONTEXTS_LIST_IDS[0]

[(0.0, 38), (0.0, 20952), (0.0, 3648), (0.0, 819), (0.0, 24299)]

### Готовим промпт

In [11]:
chunks_df = pd.read_csv(CHUNKS_PATH)

In [12]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    rel_score = CONTEXTS_LIST_IDS[i][0][0]
    rel_doc = chunks_df['chunk'][CONTEXTS_LIST_IDS[i][0][1]]
    documents_list = [PARAMS['item_format'].format(score=rel_score, document=rel_doc)]
    
    for raw_item in CONTEXTS_LIST_IDS[i][1:]:
        doc_chunk = chunks_df[chunks_df['index'] == raw_item[1]].reset_index(drop=True)['chunk'][0]
        formated_item = PARAMS['item_format'].format(score=raw_item[0], document=doc_chunk)
        documents_list.append(formated_item)
    documents_list = '\n'.join(documents_list)
    
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=dataset_df['question'][i]))

100%|██████████| 2000/2000 [00:15<00:00, 130.29it/s]


In [13]:
print(USER_PROMPTS[0])

Answer the question using the available information from the texts in the list below. Each text has a corresponding real-value score of its relevance to the question in square brackets at the beginning. Scores are ranged from 0.0 (the text is not suitable for generating an answer based on it) to 1.0 (the text is suitable for generating an answer based on it). Use this information. Choose texts with high enough relevance scores. If, based on the specified scores, there are no texts in the list that are relevant enough to generate answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Available information:
- [0.0] The Lost and the Plunderers '' March 4 , 2018 TBD TBD TBD TBD TBD TBD 11 `` Dead or Alive Or '' March 11 , 2018 TBD TBD TBD TBD TBD TBD 12 `` The 

In [14]:
with open(f"./logs/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"./logs/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

### Генерируем ответы на вопросы

In [15]:
generate_answers = []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
    generate_answers.append(pred_answer)

    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['short_answer'][i]}")
e_time = time()

  0%|          | 1/2000 [00:00<27:30,  1.21it/s]


[0]: 
GEN: I do not have an answer to your question.
GOLD: March 18, 2018


  5%|▌         | 101/2000 [01:15<21:01,  1.50it/s]


[100]: 
GEN: I do not have an answer to your question.
GOLD: gasoline-electric


 10%|█         | 201/2000 [02:24<19:52,  1.51it/s]

...

 90%|█████████ | 1801/2000 [21:18<02:24,  1.38it/s]


[1800]: 
GEN: I do not have an answer to your question.
GOLD: Tyra Banks


 95%|█████████▌| 1901/2000 [22:28<01:11,  1.39it/s]


[1900]: 
GEN: I do not have an answer to your question.
GOLD: large vein that carries deoxygenated blood from the lower and middle body into the right atrium of the heart.


100%|██████████| 2000/2000 [23:37<00:00,  1.41it/s]


In [16]:
# сохраняем конфигурацию эксперимента
with open(f"./logs/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    cur_item = {'gen_answer': generate_answers[i], 'used_contexts': CONTEXTS_LIST_IDS[i]}
    gen_info.append(cur_item)

with open(f"./logs/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"./logs/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [21]:
LOADING_VERSION = "2"

In [22]:
with open(f'./logs/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [23]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [24]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='en_electra_base')

Loading Meteor...
Loading ExactMatch


In [25]:
dataset_df = pd.read_csv(BASE_DATASET_PATH)

In [26]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL':[]}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL':[]}

show_step = 100

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['short_answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])

    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

  0%|          | 0/2000 [00:00<?, ?it/s]/home/jovyan/work/alexander_workspace/conda/envs/rag_alex/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/envs/rag_alex/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [09:11<00:00,  3.63it/s, BLEU2=0.879, BLEU1=0.886, ExactMatch=0.996, METEOR=0.985, BertScore=nan, Levenshtain=1.28, ROUGEL=0.996]


In [27]:
with open(f"./logs/v{PARAMS['version']}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))